# HR Employee Attrition - Feature Engineering

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")

## 2. Load Cleaned Data

In [2]:
# Load the cleaned data from previous step
df = pd.read_csv('../data/cleaned_hr_attrition.csv')

print("Dataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Dataset Shape: (1470, 32)

Columns: ['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


## 3. Split Data into Training and Testing Sets

**Important:** Always split before feature engineering to prevent data leakage.

In [3]:
# Separate features and target
X = df.drop('Attrition', axis=1)
y = df['Attrition']

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print(f"\nTraining target distribution:\n{y_train.value_counts(normalize=True) * 100}")
print(f"\nTest target distribution:\n{y_test.value_counts(normalize=True) * 100}")

Training set shape: (1176, 31)
Test set shape: (294, 31)

Training target distribution:
Attrition
No     83.843537
Yes    16.156463
Name: proportion, dtype: float64

Test target distribution:
Attrition
No     84.013605
Yes    15.986395
Name: proportion, dtype: float64


## 4. Feature Engineering

### 4.1 Create New Features

In [4]:
def engineer_features(df):
    """
    Create new features from existing ones.
    This function can be applied to both training and test sets.
    """
    df_eng = df.copy()
    
    # 1. Experience Ratio: Proportion of time at current company
    df_eng['ExperienceRatio'] = df_eng['YearsAtCompany'] / (df_eng['TotalWorkingYears'] + 1e-5)
    
    # 2. High Education Flag (Education >= 4)
    df_eng['IsHighEducation'] = (df_eng['Education'] >= 4).astype(int)
    
    # 3. Seniority Flag (JobLevel >= 2)
    df_eng['IsSenior'] = (df_eng['JobLevel'] >= 2).astype(int)
    
    # 4. High Job Involvement Flag (JobInvolvement >= 3)
    df_eng['HighJobInvolvement'] = (df_eng['JobInvolvement'] >= 3).astype(int)
    
    # 5. Tenure Category (categorical grouping of YearsAtCompany)
    df_eng['TenureCategory'] = pd.cut(df_eng['YearsAtCompany'], 
                                     bins=[-1, 1, 5, 10, 20, 100],
                                     labels=['<1 year', '1-5 years', '5-10 years', '10-20 years', '20+ years'])
    
    # 6. Age Group (categorical grouping of Age)
    df_eng['AgeGroup'] = pd.cut(df_eng['Age'], 
                               bins=[17, 25, 35, 45, 55, 100],
                               labels=['18-25', '26-35', '36-45', '46-55', '55+'])
    
    return df_eng

# Apply feature engineering to training and test sets
X_train_eng = engineer_features(X_train)
X_test_eng = engineer_features(X_test)

print("New features created:")
new_features = ['ExperienceRatio', 'IsHighEducation', 'IsSenior', 
                'HighJobInvolvement', 'TenureCategory', 'AgeGroup']
print(new_features)

New features created:
['ExperienceRatio', 'IsHighEducation', 'IsSenior', 'HighJobInvolvement', 'TenureCategory', 'AgeGroup']


### 4.2 Verify New Features

In [5]:
# Check the new features
print("Sample of engineered features (Training):")
print(X_train_eng[new_features].head())

print("\nDescription of ExperienceRatio:")
print(X_train_eng['ExperienceRatio'].describe())

print("\nDistribution of TenureCategory:")
print(X_train_eng['TenureCategory'].value_counts())

Sample of engineered features (Training):
      ExperienceRatio  IsHighEducation  IsSenior  HighJobInvolvement  \
1194         0.103448                1         1                   1   
128          0.666664                0         0                   1   
810          0.521739                0         1                   1   
478          0.999999                0         0                   1   
491          0.799999                1         1                   1   

     TenureCategory AgeGroup  
1194      1-5 years    46-55  
128       1-5 years    18-25  
810     10-20 years    46-55  
478      5-10 years    18-25  
491      5-10 years    36-45  

Description of ExperienceRatio:
count    1176.000000
mean        0.680378
std         0.326258
min         0.000000
25%         0.416666
50%         0.799998
75%         0.999995
max         1.000000
Name: ExperienceRatio, dtype: float64

Distribution of TenureCategory:
TenureCategory
1-5 years      449
5-10 years     364
<1 year       

## 5. Encode Categorical Variables

### 5.1 Identify Column Types

In [6]:
# Identify categorical columns
categorical_cols = X_train_eng.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)

# Separate nominal and ordinal
nominal_cols = ['BusinessTravel', 'Department', 'EducationField', 
                'Gender', 'MaritalStatus', 'OverTime', 'TenureCategory', 'AgeGroup']

ordinal_cols = ['EnvironmentSatisfaction', 'JobSatisfaction', 
                'RelationshipSatisfaction', 'WorkLifeBalance']

print(f"\nNominal columns: {nominal_cols}")
print(f"Ordinal columns: {ordinal_cols}")

Categorical columns: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']

Nominal columns: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'MaritalStatus', 'OverTime', 'TenureCategory', 'AgeGroup']
Ordinal columns: ['EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction', 'WorkLifeBalance']


### 5.2 Encode Ordinal Columns (Label Encoding)

In [7]:
# Create a copy of the data
X_train_encoded = X_train_eng.copy()
X_test_encoded = X_test_eng.copy()

# Encode ordinal columns
for col in ordinal_cols:
    # Create a LabelEncoder for each column
    le = LabelEncoder()
    # Fit on training data
    X_train_encoded[f'{col}_encoded'] = le.fit_transform(X_train_eng[col])
    # Transform test data
    X_test_encoded[f'{col}_encoded'] = le.transform(X_test_eng[col])
    
    print(f"Encoding {col}: {le.classes_} -> {le.transform(le.classes_)}")

# Drop original ordinal columns
X_train_encoded = X_train_encoded.drop(columns=ordinal_cols)
X_test_encoded = X_test_encoded.drop(columns=ordinal_cols)

Encoding EnvironmentSatisfaction: [1 2 3 4] -> [0 1 2 3]
Encoding JobSatisfaction: [1 2 3 4] -> [0 1 2 3]
Encoding RelationshipSatisfaction: [1 2 3 4] -> [0 1 2 3]
Encoding WorkLifeBalance: [1 2 3 4] -> [0 1 2 3]


### 5.3 Encode Nominal Columns (One-Hot Encoding)

In [8]:
# Use pandas get_dummies for one-hot encoding
X_train_encoded = pd.get_dummies(X_train_encoded, columns=nominal_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_encoded, columns=nominal_cols, drop_first=True)

# Align test columns with training columns
X_train_cols = X_train_encoded.columns
X_test_encoded = X_test_encoded.reindex(columns=X_train_cols, fill_value=0)

print(f"Shape after encoding: {X_train_encoded.shape}")
print(f"Number of features: {len(X_train_encoded.columns)}")
print(f"\nFirst few columns:\n{X_train_encoded.columns[:10].tolist()}")

Shape after encoding: (1176, 50)
Number of features: 50

First few columns:
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeNumber', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'MonthlyIncome']


## 6. Scale Numerical Features

In [9]:
# Identify numerical columns (excluding encoded categorical ones)
# We'll scale all remaining numeric columns that aren't already encoded
numeric_cols = X_train_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in ['EmployeeNumber']]  # Exclude ID-like columns

print(f"Numerical columns to scale: {len(numeric_cols)} columns")

# Scale the numerical features
scaler = StandardScaler()

# Fit on training data
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train_encoded[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test_encoded[numeric_cols])

print("\nVerification - Mean and Std after scaling (training):")
print(X_train_scaled[numeric_cols].describe().loc[['mean', 'std']].round(2))

Numerical columns to scale: 24 columns

Verification - Mean and Std after scaling (training):
      Age  DailyRate  DistanceFromHome  Education  HourlyRate  JobInvolvement  \
mean -0.0        0.0              -0.0        0.0        -0.0             0.0   
std   1.0        1.0               1.0        1.0         1.0             1.0   

      JobLevel  MonthlyIncome  MonthlyRate  NumCompaniesWorked  ...  \
mean       0.0           -0.0          0.0                -0.0  ...   
std        1.0            1.0          1.0                 1.0  ...   

      TrainingTimesLastYear  YearsAtCompany  YearsInCurrentRole  \
mean                    0.0            -0.0                 0.0   
std                     1.0             1.0                 1.0   

      YearsSinceLastPromotion  YearsWithCurrManager  ExperienceRatio  \
mean                     -0.0                   0.0              0.0   
std                       1.0                   1.0              1.0   

      EnvironmentSatisfaction

## 7. Verify the Final Processed Data

In [10]:
# Check final shapes
print(f"X_train_final shape: {X_train_scaled.shape}")
print(f"X_test_final shape: {X_test_scaled.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Check for any remaining categorical columns
remaining_object_cols = X_train_scaled.select_dtypes(include=['object']).columns.tolist()
if remaining_object_cols:
    print(f"\nWarning: Still have object columns: {remaining_object_cols}")
else:
    print("\n✅ All columns are now numeric!")

# Save the preprocessed data
X_train_scaled.to_csv('../data/X_train_processed.csv', index=False)
X_test_scaled.to_csv('../data/X_test_processed.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print("\n✅ Processed data saved successfully!")

X_train_final shape: (1176, 50)
X_test_final shape: (294, 50)
y_train shape: (1176,)
y_test shape: (294,)


✅ Processed data saved successfully!


## 8. Feature Engineering Summary

In [11]:
# Create a summary of changes
summary = {
    'Action': [
        'Original Features',
        'New Features Added',
        'Ordinal Encoding',
        'One-Hot Encoding',
        'Feature Scaling',
        'Final Features'
    ],
    'Count': [
        len(X.columns),
        len(new_features),
        len(ordinal_cols),
        len(nominal_cols),
        len(numeric_cols),
        X_train_scaled.shape[1]
    ],
    'Details': [
        'Original 32 features',
        f'Added {len(new_features)} engineered features',
        f'Encoded {len(ordinal_cols)} ordinal features',
        f'Encoded {len(nominal_cols)} nominal features',
        f'Scaled {len(numeric_cols)} numerical features',
        f'Total features ready for modeling'
    ]
}

summary_df = pd.DataFrame(summary)
print("Feature Engineering Summary:")
print(summary_df)

# Save feature engineering details
feature_cols = pd.DataFrame({
    'Feature': X_train_scaled.columns.tolist(),
    'Type': ['numeric' if col not in nominal_cols and '_encoded' not in col else 'encoded' for col in X_train_scaled.columns]
})
feature_cols.to_csv('../data/feature_columns.csv', index=False)
print("\n✅ Feature columns saved to 'data/feature_columns.csv'")

Feature Engineering Summary:
               Action  Count                            Details
0   Original Features     31               Original 32 features
1  New Features Added      6        Added 6 engineered features
2    Ordinal Encoding      4         Encoded 4 ordinal features
3    One-Hot Encoding      8         Encoded 8 nominal features
4     Feature Scaling     24       Scaled 24 numerical features
5      Final Features     50  Total features ready for modeling

✅ Feature columns saved to 'data/feature_columns.csv'
